# EMA 55 Crossover Walk-Forward Optimizer
# Training: Pre-2023 | Testing: 2023+
# Optimization: Target Exits, Stop Losses, Time Stops

In [1]:
# 1. Imports and Setup
import pandas as pd
import numpy as np
import os
import glob
import random
import itertools
from IPython.display import display, HTML
import warnings

warnings.filterwarnings('ignore')
display(HTML("<style>.container { width:100% !important; }</style>"))

DATA_DIR = r'D:\0dot1_Aug_2016_master\data\mstock_mtf_daily_data'
TRAIN_END_DATE = '2023-01-01'


In [2]:
# 2. Data Loading & Indicator Calculations
def calc_rsi(series, period):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

all_data = []
files = glob.glob(os.path.join(DATA_DIR, '*.csv'))
random.seed(42) 
random.shuffle(files)

loaded = 0
for file_path in files:
    if loaded >= 50: 
        break
    symbol = os.path.basename(file_path).replace('.csv', '')
    try:
        df = pd.read_csv(file_path).dropna(subset=['Close'])
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values('Date').reset_index(drop=True)
        
        if len(df) < 500: 
            continue
            
        latest_price = df['Close'].iloc[-1]
        if not (90 <= latest_price <= 600):
            continue
            
        # Core Indicators
        df['EMA_55'] = df['Close'].ewm(span=55, adjust=False).mean()
        df['EMA_221'] = df['Close'].ewm(span=221, adjust=False).mean()
        df['RSI_5'] = calc_rsi(df['Close'], 5)
        
        # Crossover Logic
        df['Prev_Close'] = df['Close'].shift(1)
        df['Prev_EMA_55'] = df['EMA_55'].shift(1)
        
        # Entry Filters
        # 1. EMA 55 Crossover
        crossover = (df['Prev_Close'] < df['Prev_EMA_55']) & (df['Close'] >= df['EMA_55'])
        # 2. Regime Filter
        regime = df['Close'] > df['EMA_221']
        # 3. Fresh Trend Filter
        fresh_trend = ((df['EMA_55'] - df['EMA_221']) / df['EMA_221']) <= 0.08
        # 4. Short-Term Overbought Filter
        not_overbought = df['RSI_5'] <= 68
        
        df['Setup_Signal'] = crossover & regime & fresh_trend & not_overbought
        
        df = df.tail(1000).reset_index(drop=True)
        df['Symbol'] = symbol
        all_data.append(df)
        loaded += 1
    except Exception as e:
        pass

data = pd.concat(all_data, ignore_index=True)
print(f"Loaded and calculated {len(data['Symbol'].unique())} stocks.")


Loaded and calculated 50 stocks.


In [3]:
# 3. Vectorized Backtest Engine
def backtest_ema(df, take_profit, stop_loss, time_stop):
    trades = []
    in_trade = False
    entry_price = 0
    entry_date = None
    days_held = 0
    
    for i, row in df.iterrows():
        if pd.isna(row['EMA_55']):
            continue
            
        if in_trade:
            days_held += 1
            exit_triggered = False
            exit_reason = ""
            
            profit_pct = (row['Close'] - entry_price) / entry_price
            
            if profit_pct >= take_profit:
                exit_triggered = True
                exit_reason = "Take Profit"
            elif profit_pct <= stop_loss:
                exit_triggered = True
                exit_reason = "Stop Loss"
            elif days_held >= time_stop:
                exit_triggered = True
                exit_reason = "Time Stop"
            
            if exit_triggered:
                trades.append({
                    'Symbol': row['Symbol'],
                    'Entry_Date': entry_date,
                    'Exit_Date': row['Date'],
                    'Return': profit_pct,
                    'Reason': exit_reason
                })
                in_trade = False
                days_held = 0
                
        if not in_trade and row['Setup_Signal']:
            in_trade = True
            entry_price = row['Close']
            entry_date = row['Date']
                        
    return trades


In [4]:
# 4. Grid Search Optimizer
# Grid Space (Single Setup Test)
TAKE_PROFIT = [0.05]
STOP_LOSS = [-0.99] # Effectively no stop loss
TIME_STOP = [8]

train_data = data[data['Date'] < TRAIN_END_DATE].copy()
test_data = data[data['Date'] >= TRAIN_END_DATE].copy()

leaderboard = []

total_combinations = len(TAKE_PROFIT) * len(STOP_LOSS) * len(TIME_STOP)
print(f"Starting Walk-Forward Optimization. Grid size: {total_combinations} combinations...")

run_count = 0
for tp, sl, ts in itertools.product(TAKE_PROFIT, STOP_LOSS, TIME_STOP):
    run_count += 1
        
    # Phase 1: Train
    train_trades = []
    for sym, grp in train_data.groupby('Symbol'):
        train_trades.extend(backtest_ema(grp.reset_index(drop=True), tp, sl, ts))
        
    train_df = pd.DataFrame(train_trades)
    
    if len(train_df) < 10: continue
    train_win_rate = (train_df['Return'] > 0).mean()
    if train_win_rate < 0.40: continue # Allow slightly lower win rates for trend following
        
    # Phase 2: Test on Unseen Data (2023+)
    test_trades = []
    for sym, grp in test_data.groupby('Symbol'):
        test_trades.extend(backtest_ema(grp.reset_index(drop=True), tp, sl, ts))
        
    test_df = pd.DataFrame(test_trades)
    
    if len(test_df) < 5: continue
        
    test_win_rate = (test_df['Return'] > 0).mean()
    test_avg_ret = test_df['Return'].mean()
    
    if test_avg_ret <= 0: continue
        
    # Phase 3: Calculate Fitness
    test_df = test_df.sort_values('Exit_Date').reset_index(drop=True)
    equity = 10000 * (1 + test_df['Return']).cumprod()
    drawdown = ((equity / equity.cummax()) - 1).min()
    drawdown = abs(drawdown) if drawdown < 0 else 0.01
    
    fitness = (test_win_rate * test_avg_ret) / drawdown
    
    params_str = f"TP:+{int(tp*100)}% | SL:{int(sl*100)}% | TS:{ts}d"
    
    leaderboard.append({
        'Params': params_str,
        'Test_Trades': len(test_df),
        'Test_WinRate': test_win_rate * 100,
        'Test_AvgReturn': test_avg_ret * 100,
        'Test_Drawdown': drawdown * 100,
        'Fitness': fitness
    })

print("Optimization Complete.")


Starting Walk-Forward Optimization. Grid size: 1 combinations...


Optimization Complete.


In [5]:
# 5. Leaderboard Showdown
if len(leaderboard) > 0:
    leaderboard_df = pd.DataFrame(leaderboard).sort_values('Fitness', ascending=False).reset_index(drop=True)
    
    display(HTML("<h3>Leaderboard Showdown (Tested on Unseen 2023+ Data)</h3>"))
    
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: lightgreen' if v else '' for v in is_max]
        
    styled_df = leaderboard_df.head(15).style.apply(highlight_max, subset=['Fitness', 'Test_WinRate', 'Test_AvgReturn']).format({
        'Test_WinRate': "{:.2f}%",
        'Test_AvgReturn': "{:.2f}%",
        'Test_Drawdown': "{:.2f}%",
        'Fitness': "{:.4f}"
    })
    
    display(styled_df)
    
    print("\n🏆 CHAMPION STRATEGY:")
    print("=============================================")
    print(leaderboard_df.iloc[0]['Params'])
else:
    print("No strategies survived the Walk-Forward Optimization (No profitable setups found on unseen data).")


,Params,Test_Trades,Test_WinRate,Test_AvgReturn,Test_Drawdown,Fitness
0,TP:+5% | SL:-99% | TS:8d,276,55.43%,0.82%,37.10%,0.0122



🏆 CHAMPION STRATEGY:
TP:+5% | SL:-99% | TS:8d
